In [9]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/")
RAW_PATH = PROJECT_ROOT / "UgandaLSMS" / "Wave_3" / "gsec2.dta"

OUT_ROOT = PROJECT_ROOT / "Finished sections" / "Household"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

OUT_CSV = OUT_ROOT / "GSEC15A_wave3_from_GSEC2.csv"
OUT_XLSX = OUT_ROOT / "GSEC15A_wave3_from_GSEC2_preview.xlsx"
OUT_PARQUET = OUT_ROOT / "GSEC15A_wave3_from_GSEC2.parquet"

PRESENT_MONTHS_MIN = 6
ADULT_AGE_MIN = 18

df = pd.read_stata(RAW_PATH, convert_categoricals=False)
df.columns = df.columns.str.strip().str.upper()

needed = ["HHID", "PID", "H2Q1", "H2Q3", "H2Q4", "H2Q5", "H2Q7", "H2Q8"]
missing = [c for c in needed if c not in df.columns]

if missing:
    raise KeyError(f"Missing columns in GSEC2: {missing}")

female_code_by_wave = {
    2: 0,
    3: 2,
}

WAVE = 3  # change to 2 or 3

df = df[needed].copy()

df["HHID"] = df["HHID"].astype("string").str.strip()
df["PID"] = df["PID"].astype("string").str.strip()
df["RELATIONSHIP"] = pd.to_numeric(df["H2Q4"], errors="coerce")
df["MONTHS_PRESENT"] = pd.to_numeric(df["H2Q5"], errors="coerce")
df["AGE"] = pd.to_numeric(df["H2Q8"], errors="coerce")
df["SEX"] = pd.to_numeric(df["H2Q3"], errors="coerce")


df["MALE"] = df["SEX"].eq(1)
df["FEMALE"] = df["SEX"].eq(female_code_by_wave[WAVE])

df["IS_CORE_FAMILY"] = df["RELATIONSHIP"].isin([1, 2])

df["PRESENT_FAMILY_MEMBER"] = (
    df["MONTHS_PRESENT"].ge(PRESENT_MONTHS_MIN)
    | df["IS_CORE_FAMILY"]
)

df["ADULT"] = df["AGE"].ge(ADULT_AGE_MIN)
df["CHILD"] = df["AGE"].lt(ADULT_AGE_MIN)

df_use = df[df["PRESENT_FAMILY_MEMBER"]].copy()

df_use["H15A1"] = (df_use["ADULT"] & df_use["MALE"]).astype(int)
df_use["H15A2"] = (df_use["ADULT"] & df_use["FEMALE"]).astype(int)
df_use["H15A3"] = (df_use["CHILD"] & df_use["MALE"]).astype(int)
df_use["H15A4"] = (df_use["CHILD"] & df_use["FEMALE"]).astype(int)

gsec15a_wave2 = (
    df_use
    .groupby("HHID", dropna=False)[["H15A1", "H15A2", "H15A3", "H15A4"]]
    .sum()
    .reset_index()
)

all_hhids = df[["HHID"]].drop_duplicates()

gsec15a_wave2 = (
    all_hhids
    .merge(gsec15a_wave2, on="HHID", how="left")
    .fillna({"H15A1": 0, "H15A2": 0, "H15A3": 0, "H15A4": 0})
)

for col in ["H15A1", "H15A2", "H15A3", "H15A4"]:
    gsec15a_wave2[col] = gsec15a_wave2[col].astype(int)

gsec15a_wave2.insert(0, "WAVE", 3)
gsec15a_wave2.insert(1, "SOURCE_FILE", "GSEC2.dta")
gsec15a_wave2.insert(2, "SOURCE_SECTION", "GSEC2_DERIVED_FOR_GSEC15A")

# Troubleshoot check
gsec15a_wave2["TOTAL_MEMBERS_DERIVED"] = (
    gsec15a_wave2["H15A1"]
    + gsec15a_wave2["H15A2"]
    + gsec15a_wave2["H15A3"]
    + gsec15a_wave2["H15A4"]
)

display(gsec15a_wave2[gsec15a_wave2["HHID"] == "1013000201"])

gsec15a_wave2.to_csv(OUT_CSV, index=False)
gsec15a_wave2.to_excel(OUT_XLSX, index=False)
gsec15a_wave2.to_parquet(OUT_PARQUET, index=False)

print(f"Rows exported: {len(gsec15a_wave2):,}")
print(f"CSV: {OUT_CSV}")
print(f"Excel: {OUT_XLSX}")
print(f"Parquet: {OUT_PARQUET}")

,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,H15A1,H15A2,H15A3,H15A4,TOTAL_MEMBERS_DERIVED
0,3,GSEC2.dta,GSEC2_DERIVED_FOR_GSEC15A,1013000201,1,2,1,0,4


Rows exported: 2,850
CSV: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15A_wave3_from_GSEC2.csv
Excel: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15A_wave3_from_GSEC2_preview.xlsx
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15A_wave3_from_GSEC2.parquet
